# FedKG-Maint — Phase B · Development-Only Improvement Study (v1)

**Scope:** Microsoft Azure PdM, existing completed `unified_v3_final_20261025` run. This notebook **does not modify** the existing v3 artifacts.

This is **one experiment notebook** with two connected research tasks:

1. **B1 — HGB feature ablations and actionable early-warning analysis.** Use four development factories only; quantify dependence on maintenance/error history using 4-fold leave-one-factory-out (LOFO). Measure event-level warnings before failure at fixed alert thresholds.
2. **B2 — Matched federated MLP vs temporal LSTM.** Same available factories, same training endpoints, identical held-out development rows, equal client weighting and fixed training budget. LSTM reads **past/current 24-hour feature sequences**, while MLP reads only the current feature vector.

**Scientific boundaries**
- All model selection happens on **factories 1–4**. `factory_5` is never loaded in this notebook.
- This is **development evidence**, not a new blind external-site test; factory 5 was examined in earlier runs.
- We reuse the original v3 25 features and client mapping; **no new feature/scaler fitting**.
- The LSTM/FedAvg comparison is **non-private**. Existing DP-SGD guarantees **do not transfer** to these newly trained networks.
- Early-warning lead times are based on recorded failure timestamps; they do **not** prove that an alarm would have prevented failure.
- Do not claim that a maintenance feature has caused an outcome. Ablations establish predictive dependence, not causality.
- NASA C-MAPSS and LLM ablations are separate later stages of Plan B.

**Execution:** Upload/open this notebook in Google Colab, mount Drive, run all cells. It expects the previous v3 run at the specified directory. Runtime: GPU is preferable for the optional federated LSTM experiment. No API key is needed.

In [ ]:
import os, json, hashlib, warnings, random, copy, time, sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
warnings.filterwarnings("ignore", category=FutureWarning)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

DRIVE = Path(os.environ.get("FEDKG_DRIVE_ROOT", "/content/drive/MyDrive/FedKG-Maint")).expanduser()
V3 = DRIVE / "unified_runs" / "unified_v3_final_20261025"
PREP = V3 / "prep_v3"
PROC = PREP / "processed"
CONF = PREP / "configs"
SCALER_DIR = PREP / "scalers"
RAW = DRIVE / "data" / "raw"

# Change the tag when intentionally starting a distinct new study. Original v3 is read-only.
B_TAG = os.environ.get("FEDKG_B_TAG", "phaseB_development_v1")
OUT = DRIVE / "unified_runs" / B_TAG
assert (V3 / "RUN_COMPLETE.json").exists(), "Completed unified v3 run missing. Do not regenerate or guess its contents."
assert (PROC / "train.parquet").exists() and (PROC / "validation.parquet").exists()
assert (PREP / "manifests" / "factory_manifest.csv").exists()
assert (RAW / "PdM_failures.csv").exists()
assert V3.resolve() != OUT.resolve()
OUT.mkdir(parents=True, exist_ok=True)
assert not (OUT / "B_COMPLETE.json").exists(), "This B study is already closed. Change FEDKG_B_TAG for a new study."
for sub in ("results", "reports", "logs"):
    (OUT / sub).mkdir(exist_ok=True)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for block in iter(lambda: f.read(1<<20), b""):
            h.update(block)
    return h.hexdigest()

SOURCE_HASHES = {
    "factory_manifest.csv": sha256(PREP/"manifests"/"factory_manifest.csv"),
    "feature_groups.json": sha256(CONF/"feature_groups.json"),
    "train.parquet": sha256(PROC/"train.parquet"),
    "validation.parquet": sha256(PROC/"validation.parquet"),
}
print("Source (never modified):", V3)
print("New output directory:", OUT)
print("Checksums pinned:", SOURCE_HASHES)

## B0 — Load the **same** v3 dataset protocol

Pre-registered conditions: four development factories (`factory_1`–`factory_4`), original v3 fixed mixed-model partition, `label_failure_24h`, `train`/`validation` with the existing purge. No site-5 records are read.

In [ ]:
CONFIG = json.loads((CONF/"feature_groups.json").read_text())
FEATURES = list(CONFIG["feature_sets"]["full"])
assert len(FEATURES) == 25 and len(set(FEATURES)) == 25
SENSORS = ["volt", "rotate", "pressure", "vibration"]
DEV = [f"factory_{i}" for i in range(1, 5)]
LABEL = "label_failure_24h"
KEYS = ["machineID", "datetime", "client_id"]

TR = pd.read_parquet(PROC / "train.parquet")
VA = pd.read_parquet(PROC / "validation.parquet")
manifest = pd.read_csv(PREP/"manifests"/"factory_manifest.csv")
failure_events = pd.read_csv(RAW/"PdM_failures.csv", parse_dates=["datetime"])
failure_events = failure_events[["machineID","datetime"]].drop_duplicates().sort_values(["machineID","datetime"])
for name, df in [("train", TR), ("validation", VA)]:
    assert set(df["client_id"]) == set(DEV), f"{name}: unexpected clients"
    assert "factory_5" not in set(df["client_id"])
    assert not df.duplicated(["machineID","datetime"]).any()
    assert set(FEATURES + [LABEL] + KEYS) <= set(df.columns)
    assert df[FEATURES].notna().all().all() and np.isfinite(df[FEATURES].to_numpy()).all()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df.sort_values(["machineID","datetime"], inplace=True)
    df.reset_index(drop=True, inplace=True)
assert set(TR.machineID).isdisjoint(set(manifest.loc[manifest.client_id.eq("factory_5"), "machineID"]))
assert set(VA.machineID).isdisjoint(set(manifest.loc[manifest.client_id.eq("factory_5"), "machineID"]))
assert (TR.datetime.max() + pd.Timedelta(hours=24) < VA.datetime.min())
assert (manifest.groupby("client_id").machineID.nunique() == 20).all()

feature_groups = {
    "full": FEATURES,
    "no_maintenance": [x for x in FEATURES if x != "hours_since_last_maintenance_log1p"],
    "sensor_errors": [x for x in FEATURES if x in SENSORS or "_past_" in x or
                      x.startswith("past_observation_count") or
                      x in ("has_prior_error","hours_since_last_error_log1p")],
    "sensor_history_only": [x for x in FEATURES if x in SENSORS or "_past_" in x or
                            x.startswith("past_observation_count")],
}
assert [len(feature_groups[k]) for k in feature_groups] == [25,24,24,22]
assert feature_groups["sensor_history_only"] == [
    x for x in FEATURES if x not in
    {"hours_since_last_error_log1p","has_prior_error","hours_since_last_maintenance_log1p"}
]
FACTORY_COUNTS = pd.DataFrame({
    "train_rows": TR.groupby("client_id").size(),
    "validation_rows": VA.groupby("client_id").size(),
    "train_prevalence": TR.groupby("client_id")[LABEL].mean(),
    "val_prevalence": VA.groupby("client_id")[LABEL].mean(),
})
display(FACTORY_COUNTS)
print("Feature groups:", {k:len(v) for k,v in feature_groups.items()})
print("Dataset locked. No factory_5 evaluation imported.")

## B1 — HGB ablation with 4-fold leave-one-factory-out

For each held-out development factory, fit on the **other three factories' training segments** and predict **that held-out factory's validation segment**. HGB inputs remain in physical units (not target-factory fitted statistics).

Training cap and HGB hyperparameters follow the v3 development HGB experiment: 25,000 training rows per source client, 90 boosting iterations, depth 3, learning rate 0.08, L2=0.2. Same client subsample for **all four feature variants** within a fold.

**No tuning on Factory 5.** Save fold-level predictions for operational audits.

In [ ]:
HGB_CAP = 25_000
HGB_SPECS = {
    "max_iter": 90, "max_depth": 3, "learning_rate": 0.08,
    "l2_regularization": 0.2, "random_state": 42
}
hgb_metrics = []
hgb_predictions = {}
for fold, held in enumerate(DEV):
    sources = [c for c in DEV if c != held]
    # Deterministic, same training rows for every ablation in this fold.
    rows = []
    for j, c in enumerate(sources):
        pool = TR.index[TR.client_id.eq(c)].to_numpy()
        rng = np.random.default_rng(4100 + 31*fold + 7*j)
        selected = np.sort(rng.choice(pool, size=min(HGB_CAP, len(pool)), replace=False))
        rows.append(selected)
    fit_rows = np.sort(np.concatenate(rows))
    valid_rows = VA.index[VA.client_id.eq(held)].to_numpy()
    trainfold, valfold = TR.iloc[fit_rows], VA.iloc[valid_rows]
    yy = valfold[LABEL].to_numpy(dtype=np.int8)
    for variant, feat in feature_groups.items():
        clf = HistGradientBoostingClassifier(**HGB_SPECS)
        clf.fit(trainfold[feat].to_numpy(dtype=np.float32),
                trainfold[LABEL].to_numpy(dtype=np.int8))
        p = clf.predict_proba(valfold[feat].to_numpy(dtype=np.float32))[:,1].astype(np.float32)
        name = f"hgb_raw_{variant}"
        hgb_predictions[(held,variant)] = p
        hgb_metrics.append({
            "fold": held, "variant": variant, "train_clients": ",".join(sources),
            "n_train": len(trainfold), "n_val": len(valfold), "val_prevalence": float(yy.mean()),
            "pr_auc": float(average_precision_score(yy,p)),
            "roc_auc": float(roc_auc_score(yy,p)),
        })
    pd.DataFrame(hgb_metrics).to_csv(OUT/"results"/"B1_hgb_lofo_fold_metrics.csv",index=False)
    print("B1 completed:", held, "| variants:", len(feature_groups))

HGB_RESULTS = pd.DataFrame(hgb_metrics)
HGB_SUMMARY = HGB_RESULTS.groupby("variant",as_index=False).agg(
    mean_pr_auc=("pr_auc","mean"),
    worst_factory_pr_auc=("pr_auc","min"),
    std_pr_auc=("pr_auc","std"),
    mean_roc_auc=("roc_auc","mean"),
)
HGB_SUMMARY.to_csv(OUT/"results"/"B1_hgb_lofo_summary.csv",index=False)
display(HGB_SUMMARY.sort_values("mean_pr_auc",ascending=False))

## B1 continued — What would an **actionable alarm** mean?

Use predeclared risk thresholds **0.30 / 0.50** and 24-hour cooldown; inspect minimum 1-, 6-, and 12-hour advance warning. These thresholds are **not tuned** to maximize evaluation performance. They serve as an exploratory operating curve; scores may not be calibrated probabilities.

Definitions:
- Alert: a threshold-crossing row; subsequent alerts for that machine suppressed for 24h.
- Actionable true alert: the **first subsequent failure** falls between `min_lead` and 24 hours after the alert. Later failures cannot rescue an alert if an earlier failure occurs sooner.
- Eligible event: a recorded failure with at least one observable held-out validation timestamp in `[failure-24h, failure-min_lead]`.
- Event recall: unique eligible failures matched by at least one actionable alert / eligible failures.
- False alarms per machine-day: unmatched/same-too-late alerts / (observed machine-hours / 24).

No alarms from factory 5 are generated here.

In [ ]:
from collections import defaultdict

def operational_audit(meta, p, events, threshold, min_lead_hours=1, cooldown_hours=24, horizon_hours=24):
    assert 0 < min_lead_hours <= horizon_hours
    assert len(meta)==len(p) and set(["machineID","datetime"])<=set(meta)
    mdf = meta[["machineID","datetime"]].copy().reset_index(drop=True)
    mdf["score"] = np.asarray(p,dtype=float)
    mdf["datetime"] = pd.to_datetime(mdf["datetime"])
    mdf.sort_values(["machineID","datetime"], inplace=True, kind="stable")
    failure_lookup = {m: np.sort(g["datetime"].to_numpy(dtype="datetime64[ns]"))
                      for m,g in events.groupby("machineID")}
    allowed = pd.Timedelta(hours=horizon_hours)
    earliest = pd.Timedelta(hours=min_lead_hours)
    cooldown = pd.Timedelta(hours=cooldown_hours)
    total_alerts, true_alerts, detected_events, eligible_events = 0,0,set(),set()
    true_leads = []
    for mid, g in mdf.groupby("machineID",sort=False):
        times = g.datetime.to_numpy(dtype="datetime64[ns]")
        s = g.score.to_numpy(float)
        ftime = failure_lookup.get(mid,np.array([],dtype="datetime64[ns]"))
        # Denominator: events with an opportunity to issue a sufficiently early alert.
        for ft in ftime:
            start_t = ft - allowed.to_timedelta64()
            end_t = ft - earliest.to_timedelta64()
            lo = np.searchsorted(times,start_t,side="left")
            hi = np.searchsorted(times,end_t,side="right")
            if hi > lo: eligible_events.add((str(mid), pd.Timestamp(ft).isoformat()))
        next_ok = None
        for t, score in zip(times,s):
            tt = pd.Timestamp(t)
            if score < threshold or (next_ok is not None and tt < next_ok):
                continue
            total_alerts += 1
            next_ok = tt + cooldown
            k = np.searchsorted(ftime,t,side="right")
            if k < len(ftime):
                lead = pd.Timestamp(ftime[k])-tt
                if earliest <= lead <= allowed:
                    true_alerts += 1
                    detected_events.add((str(mid),pd.Timestamp(ftime[k]).isoformat()))
                    true_leads.append(lead / pd.Timedelta(hours=1))
    detected_eligible = detected_events.intersection(eligible_events)
    machine_days = len(mdf)/24.0
    return {
        "threshold": threshold, "min_lead_h": min_lead_hours, "cooldown_h": cooldown_hours,
        "total_alerts": int(total_alerts), "actionable_alerts": int(true_alerts),
        "false_alerts": int(total_alerts-true_alerts),
        "actionable_alert_precision": true_alerts/total_alerts if total_alerts else np.nan,
        "eligible_failure_events": len(eligible_events),
        "detected_failure_events": len(detected_eligible),
        "actionable_event_recall": len(detected_eligible)/len(eligible_events) if eligible_events else np.nan,
        "false_alerts_per_machine_day": (total_alerts-true_alerts)/machine_days if machine_days else np.nan,
        "median_actionable_lead_h": float(np.median(true_leads)) if true_leads else np.nan,
    }

# Self-test: a failure 12h after a 0.9 alert at the beginning of a 48-hour record.
dummy_ts = pd.date_range("2026-01-01",periods=48,freq="h")
dummy_meta = pd.DataFrame({"machineID": [999]*48,"datetime":dummy_ts})
dummy_p = np.zeros(48);dummy_p[0]=.9
dummy_events = pd.DataFrame({"machineID":[999],"datetime":[dummy_ts[12]]})
d = operational_audit(dummy_meta,dummy_p,dummy_events,.5,min_lead_hours=12)
assert d["total_alerts"]==1 and d["actionable_alerts"]==1 and d["detected_failure_events"]==1
d_bad = operational_audit(dummy_meta,dummy_p,dummy_events,.5,min_lead_hours=13)
assert d_bad["total_alerts"]==1 and d_bad["actionable_alerts"]==0
print("PASS early-warning matching self-tests.")

ops_rows = []
for held in DEV:
    val = VA.loc[VA.client_id.eq(held), ["machineID","datetime","client_id"]]
    for variant in ["full","no_maintenance","sensor_history_only"]:
        scores = hgb_predictions[(held,variant)]
        for thr in (0.30,0.50):
            for lead in (1,6,12):
                ops_rows.append({"fold":held,"variant":variant,
                                 **operational_audit(val,scores,failure_events,thr,lead)})
B1_OPS = pd.DataFrame(ops_rows)
B1_OPS.to_csv(OUT/"results"/"B1_early_warning_operational_metrics.csv",index=False)
display(B1_OPS[(B1_OPS.threshold==.5)&(B1_OPS.min_lead_h==12)].sort_values(["fold","variant"]))

## B2 — Federated temporal model: matched MLP vs LSTM

This is **controlled development research**, not a replacement of the v3 frozen selected model.

- **MLP:** 25 current-row engineered features → 32 → 8 → 1.
- **LSTM:** a 24-step window of the *same locally normalized* 25 features → LSTM hidden 32 → dense 16 → 1. Only current/past hours are included.
- **Model training:** FedAvg, 6 communication rounds, one local epoch/round, client-balanced averaging. Fixed cap of 4,000 example endpoint rows per training factory, matched between models.
- **Evaluation:** same endpoint rows within each held-out development factory for both models, same seeds.
- This experiment changes more than the neural architecture (sequence context is additional information), deliberately testing whether explicit temporal context is useful.
- **We do not add DP here.** If LSTM wins in development folds, a separate privacy experiment must be implemented and evaluated before treating LSTM as private.

You may set `RUN_B2=False` for a lightweight B1-only diagnostic run, or `SEEDS=[42]` for a first computational pilot. The default executes B2 with two seeds.

In [ ]:
RUN_B2 = True
SEEDS = [42, 43]
WINDOW = 24
FL_ROUNDS = 6
LOCAL_EPOCHS = 1
TRAIN_CAP_PER_CLIENT = 4000
BATCH_SIZE = 512
LR = 0.002
LSTM_HIDDEN = 32
assert WINDOW >= 2 and FL_ROUNDS > 0 and TRAIN_CAP_PER_CLIENT > 0
assert "factory_5" not in DEV

import torch
from torch import nn
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4,max(1,os.cpu_count() or 1)))
print("Device:",DEVICE,"| B2 enabled:",RUN_B2,"| seeds:",SEEDS)

In [ ]:
def make_scaled_rows(df):
    # Original scalers were fitted on each factory's TRAIN segment in unified v3.
    scaled = np.empty((len(df),len(FEATURES)),dtype=np.float32)
    for client in DEV:
        mask = df.client_id.eq(client).to_numpy()
        original_scaler = joblib.load(SCALER_DIR / f"local_{client}.joblib")
        scaled[mask] = original_scaler.transform(
            df.loc[mask,FEATURES].to_numpy(dtype=np.float64)).astype(np.float32)
    assert np.isfinite(scaled).all()
    return scaled

def eligible_endpoints(df, window):
    # The split is machine/time-sorted; prevent windows crossing machines or split boundaries.
    groups = df.groupby("machineID",sort=False)
    all_eligible = []
    for mid,part in groups:
        if len(part)<window: continue
        diffs = part.datetime.diff().dropna()
        assert (diffs == pd.Timedelta(hours=1)).all(), f"Unexpected temporal gap for {mid}"
        idx = part.index.to_numpy()
        assert (idx[-1]-idx[0]+1)==len(idx), "Dataframe must remain in machine-time sorted order"
        all_eligible.append(idx[window-1:])
    return np.concatenate(all_eligible).astype(np.int64)

def windows_for(scaled,endpoints,window):
    endpoints=np.asarray(endpoints,dtype=np.int64)
    offsets=np.arange(window-1,-1,-1,dtype=np.int64)
    return scaled[endpoints[:,None]-offsets[None,:]].copy()

SCALED_TR=make_scaled_rows(TR)
SCALED_VA=make_scaled_rows(VA)
END_TR=eligible_endpoints(TR,WINDOW)
END_VA=eligible_endpoints(VA,WINDOW)
assert all((TR.machineID.to_numpy()[END_TR] == TR.machineID.to_numpy()[END_TR-WINDOW+1]))
assert all((VA.machineID.to_numpy()[END_VA] == VA.machineID.to_numpy()[END_VA-WINDOW+1]))
assert set(VA.iloc[END_VA].client_id)==set(DEV)
print("Eligible endpoints: train",len(END_TR),"validation",len(END_VA))

In [ ]:
class CompactMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(25,32),nn.ReLU(),nn.Linear(32,8),
                                    nn.ReLU(),nn.Linear(8,1))
    def forward(self,x):
        return self.layers(x[:,-1,:]).squeeze(-1)

class TemporalLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(input_size=25,hidden_size=LSTM_HIDDEN,batch_first=True)
        self.head=nn.Sequential(nn.Linear(LSTM_HIDDEN,16),nn.ReLU(),nn.Linear(16,1))
    def forward(self,x):
        return self.head(self.lstm(x)[0][:,-1,:]).squeeze(-1)

def net(kind,seed):
    torch.manual_seed(seed)
    model = CompactMLP() if kind=="MLP" else TemporalLSTM()
    return model.to(DEVICE)

def train_local(global_model, x, y, seed, pos_weight):
    local=copy.deepcopy(global_model)
    local.train()
    opt=torch.optim.Adam(local.parameters(),lr=LR)
    criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(pos_weight),device=DEVICE))
    rng=np.random.default_rng(seed)
    for ep in range(LOCAL_EPOCHS):
        perm=rng.permutation(len(y))
        for a in range(0,len(perm),BATCH_SIZE):
            batch=perm[a:a+BATCH_SIZE]
            xb=torch.from_numpy(x[batch]).to(DEVICE)
            yb=torch.from_numpy(y[batch]).float().to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss=criterion(local(xb),yb)
            assert torch.isfinite(loss), "Non-finite local loss"
            loss.backward()
            nn.utils.clip_grad_norm_(local.parameters(),max_norm=5.0)  # optimization stability, NOT differential privacy
            opt.step()
    return {k:v.detach().cpu().clone() for k,v in local.state_dict().items()}

@torch.no_grad()
def predict_net(model, x):
    model.eval()
    out=[]
    for i in range(0,len(x),BATCH_SIZE):
        xb=torch.from_numpy(x[i:i+BATCH_SIZE]).to(DEVICE)
        out.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def fedavg_train(kind, datasets, seed, rounds, pos_weight):
    model=net(kind,seed)
    clients=sorted(datasets)
    assert len(clients)==3, "Only three training factories are allowed in each LOFO fold"
    for rnd in range(rounds):
        states=[]
        for j,c in enumerate(clients):
            xx,yy=datasets[c]
            states.append(train_local(model,xx,yy,
                                      seed+10007*rnd+199*j,pos_weight))
        # Both networks contain only floating trainable tensors, no BatchNorm counters.
        averaged={k:torch.stack([state[k] for state in states]).mean(0)
                  for k in states[0]}
        model.load_state_dict(averaged)
    return model

# Numerical smoke tests, no real model results.
for kind in ("MLP","LSTM"):
    fake=np.random.default_rng(7).normal(size=(32,WINDOW,25)).astype(np.float32)
    y=(fake[:,-1,0]>.3).astype(np.float32)
    model=net(kind,42)
    local_state=train_local(model,fake,y,42,2.0)
    assert set(local_state)==set(model.state_dict())
    scores=predict_net(model,fake)
    assert np.isfinite(scores).all() and np.all((0<=scores)&(scores<=1))
print("PASS local MLP and LSTM numerical smoke tests.")

In [ ]:
B2_METRICS=[]
B2_OPS=[]
B2_PREDICTIONS={}
if RUN_B2:
    for fold,held in enumerate(DEV):
        sources=[c for c in DEV if c!=held]
        # Deterministic endpoint sample, reused across architectures and seeds.
        endpoints={}
        for j,c in enumerate(sources):
            options=END_TR[TR.iloc[END_TR].client_id.eq(c).to_numpy()]
            rng=np.random.default_rng(4217+fold*19+j*7)
            chosen=np.sort(rng.choice(options,size=min(TRAIN_CAP_PER_CLIENT,len(options)),replace=False))
            endpoints[c]=chosen
        val_indices=END_VA[VA.iloc[END_VA].client_id.eq(held).to_numpy()]
        val_x=windows_for(SCALED_VA,val_indices,WINDOW)
        val_y=VA.iloc[val_indices][LABEL].to_numpy(dtype=np.int8)
        same_validation_hash=hashlib.sha256(VA.iloc[val_indices][["machineID","datetime"]]
                                              .to_csv(index=False).encode()).hexdigest()
        ds={}
        for c in sources:
            ix=endpoints[c]
            ds[c]=(windows_for(SCALED_TR,ix,WINDOW),
                   TR.iloc[ix][LABEL].to_numpy(dtype=np.float32))
        npos=sum(float(y.sum()) for _,y in ds.values())
        nobs=sum(len(y) for _,y in ds.values())
        pw=min(30.,max(1.,(nobs-npos)/max(npos,1.)))
        for kind in ("MLP","LSTM"):
            for seed in SEEDS:
                start=time.time()
                model=fedavg_train(kind,ds,seed,FL_ROUNDS,pw)
                p=predict_net(model,val_x)
                assert len(p)==len(val_y) and np.isfinite(p).all()
                record={
                    "fold":held,"architecture":kind,"method":"FedAvg",
                    "seed":seed,"rounds":FL_ROUNDS,"window_h":WINDOW,
                    "n_train":nobs,"n_validation":len(val_y),
                    "val_prevalence":float(val_y.mean()),
                    "pr_auc":float(average_precision_score(val_y,p)),
                    "roc_auc":float(roc_auc_score(val_y,p)),
                    "runtime_seconds":round(time.time()-start,1),
                    "validation_row_sha256":same_validation_hash,
                }
                B2_METRICS.append(record)
                B2_PREDICTIONS[(held,kind,seed)] = p
                # Same validation endpoints for both architectures; fixed, predeclared thresholds.
                validation_meta = VA.iloc[val_indices][["machineID", "datetime"]].reset_index(drop=True)
                for threshold in (0.30, 0.50):
                    for minimum_lead in (1, 6, 12):
                        B2_OPS.append({"fold":held, "architecture":kind, "seed":seed,
                                       **operational_audit(validation_meta,p,failure_events,
                                                           threshold,minimum_lead)})
                pd.DataFrame(B2_METRICS).to_csv(OUT/"results"/"B2_federated_temporal_fold_metrics.csv",index=False)
                pd.DataFrame(B2_OPS).to_csv(OUT/"results"/"B2_temporal_early_warning_metrics.csv",index=False)
                print(f"B2 fold={held} model={kind} seed={seed} AP={record['pr_auc']:.4f}")
                del model
        del val_x,ds
    B2_RESULTS=pd.DataFrame(B2_METRICS)
    B2_SUMMARY=B2_RESULTS.groupby("architecture",as_index=False).agg(
        mean_pr_auc=("pr_auc","mean"),worst_fold_pr_auc=("pr_auc","min"),
        std_pr_auc=("pr_auc","std"),mean_roc_auc=("roc_auc","mean")
    )
    B2_SUMMARY.to_csv(OUT/"results"/"B2_federated_temporal_summary.csv",index=False)
    B2_OPS_DF=pd.DataFrame(B2_OPS)
    display(B2_SUMMARY)
    print("Operational comparison at risk threshold 0.50 and minimum lead time 12h:")
    display(B2_OPS_DF[(B2_OPS_DF.threshold==0.5)&(B2_OPS_DF.min_lead_h==12)]
            [["fold","architecture","seed","actionable_event_recall",
              "false_alerts_per_machine_day","actionable_alert_precision"]])
else:
    print("B2 skipped by RUN_B2=False; B1 completed normally.")

## B3 — Development-only paired comparison and decision report

The paired comparison uses the **same factory/seed and evaluation endpoint rows**. Report the difference (`LSTM - MLP`), including variations across folds and seeds. This small study is not sufficient for a broad statistical significance claim.

A larger AP is **not enough by itself** to replace the model: check event-level metrics, false alarm budget, and compute/communication cost before choosing a final architecture. No new private FL guarantee is claimed.

In [ ]:
pair_report={}
if RUN_B2:
    wide=B2_RESULTS.pivot(index=["fold","seed"],columns="architecture",values="pr_auc")
    assert not wide.isna().any().any()
    wide["LSTM_minus_MLP"]=wide["LSTM"]-wide["MLP"]
    wide.reset_index().to_csv(OUT/"results"/"B3_paired_MLP_LSTM_delta.csv",index=False)
    pair_report={
        "mean_delta_pr_auc_lstm_minus_mlp":float(wide["LSTM_minus_MLP"].mean()),
        "median_delta_pr_auc":float(wide["LSTM_minus_MLP"].median()),
        "positive_pair_fraction":float((wide["LSTM_minus_MLP"]>0).mean()),
        "n_paired_folds_and_seeds":len(wide),
        "all_pairs_improved":bool((wide["LSTM_minus_MLP"]>0).all()),
        "interpretation":"Exploratory development-only LOFO; no significance or external blind-site claim"
    }
    display(wide)
    print("Paired comparison:",pair_report)

source_integrity={k:sha256(p)==SOURCE_HASHES[k] for k,p in {
    "factory_manifest.csv":PREP/"manifests"/"factory_manifest.csv",
    "feature_groups.json":CONF/"feature_groups.json",
    "train.parquet":PROC/"train.parquet",
    "validation.parquet":PROC/"validation.parquet",
}.items()}
assert all(source_integrity.values()), "Protected original v3 inputs changed during Phase B"
assert set(HGB_RESULTS.fold)==set(DEV)
assert (not RUN_B2) or set(B2_RESULTS.fold)==set(DEV)
report={
    "study":"FedKG-Maint Plan B · development-only improvement v1",
    "source_v3":str(V3),
    "source_hashes_verified_unchanged":source_integrity,
    "source_factory5_used_for_model_selection":False,
    "purpose":"diagnose maintenance/history dependence, early-warning usefulness, and temporal federated improvement",
    "client_partition":"v3 fixed mixed-model non-IID; four development factories only",
    "hgb_ablation_summary":HGB_SUMMARY.to_dict("records"),
    "early_warning_table":str(OUT/"results"/"B1_early_warning_operational_metrics.csv"),
    "temporal_federated_experiment_run":RUN_B2,
    "b2_summary":B2_SUMMARY.to_dict("records") if RUN_B2 else [],
    "b2_paired":pair_report,
    "b2_operational_table":str(OUT/"results"/"B2_temporal_early_warning_metrics.csv") if RUN_B2 else None,
    "not_demonstrated":["Fresh held-out-site performance",
                         "End-to-end differential privacy",
                         "Client-level privacy for new MLP/LSTM",
                         "End-to-end physical factory isolation",
                         "LLM explanation improvement",
                         "NASA RUL benchmark"],
    "recommended_next_stage":"Inspect development ablation and operational metrics. Select a fixed protocol on dev folds before any additional site test; separately implement and evaluate DP for the chosen architecture."
}
(OUT/"reports"/"B_development_report.json").write_text(json.dumps(report,indent=2,default=str))
lines=[
    "# FedKG-Maint Plan B — Development-only report","",
    "## B1 HGB LOFO feature ablation","",
    HGB_SUMMARY.to_markdown(index=False,floatfmt=".4f"),"",
    "## B2 Federated MLP vs LSTM","",
    (B2_SUMMARY.to_markdown(index=False,floatfmt=".4f") if RUN_B2 else "Not run"),"",
    "## Paired LSTM minus MLP","",
    (json.dumps(pair_report,indent=2) if RUN_B2 else "Not run"),"",
    "## Interpretation boundaries","",
    "- All model selection remains on development factories 1–4.",
    "- This notebook intentionally does not load the historical factory_5 evaluation.",
    "- The HGB feature ablation tests predictive dependence, not causality.",
    "- Fixed alert thresholds are exploratory and may not represent calibrated probabilities.",
    "- The MLP/LSTM runs do not provide a DP guarantee.",
    "- NASA and Phase 5 LLM evaluation are not performed here."
]
(OUT/"reports"/"B_development_report.md").write_text("\n".join(lines))
(OUT/"B_COMPLETE.json").write_text(json.dumps({
    "status":"complete","tag":B_TAG,"B1":True,"B2":RUN_B2,
    "source_hashes_verified":True
},indent=2))
print("PHASE B DEVELOPMENT COMPLETE")
print("Report:",OUT/"reports"/"B_development_report.md")
print("Fold results:",OUT/"results"/"B1_hgb_lofo_fold_metrics.csv")
if RUN_B2: print("Temporal FL:",OUT/"results"/"B2_federated_temporal_summary.csv")